# 13 — LightGBM and Random Forest tuning

This notebook tunes the two selected baseline families using training and validation data only. The classification threshold remains fixed at 0.50, and the test set is neither loaded nor evaluated.

### What this cell does
Defines the exact existing inputs and new tuning outputs, checks that test artifacts exist without loading them, and fingerprints every protected input and model artifact.

### Why it matters
Tuning must reuse the established split and preprocessing while proving that baseline models and preprocessing objects were not overwritten.

### What to understand
Only tree-preprocessed train and validation data are permitted for model fitting and comparison.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import time

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import ParameterSampler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert ROOT.name == "AdoptAI_V1"
PREPROCESSED_DIR = ROOT / "data/modeling/preprocessed"
BASELINE_MODEL_DIR = ROOT / "models/baseline"
PREPROCESSOR_DIR = ROOT / "models/preprocessing"
TUNED_MODEL_DIR = ROOT / "models/tuned"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures/model_tuning"
TUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

input_paths = {
    "X_train_tree": PREPROCESSED_DIR / "X_train_tree.csv",
    "X_validation_tree": PREPROCESSED_DIR / "X_validation_tree.csv",
    "y_train": PREPROCESSED_DIR / "y_train.csv",
    "y_validation": PREPROCESSED_DIR / "y_validation.csv",
    "validation_identifiers": PREPROCESSED_DIR / "validation_identifiers.csv",
    "baseline_metrics": REPORT_DIR / "baseline_model_comparison.csv",
    "final_split_summary": REPORT_DIR / "final_split_summary.csv",
}
protected_artifacts = {
    **input_paths,
    "baseline_lightgbm": BASELINE_MODEL_DIR / "lightgbm.joblib",
    "baseline_random_forest": BASELINE_MODEL_DIR / "random_forest.joblib",
    "tree_preprocessor": PREPROCESSOR_DIR / "tree_preprocessor.joblib",
    "linear_preprocessor": PREPROCESSOR_DIR / "linear_preprocessor.joblib",
}
test_artifacts = [PREPROCESSED_DIR / "X_test_tree.csv", PREPROCESSED_DIR / "y_test.csv"]
assert all(path.is_file() for path in protected_artifacts.values())
assert all(path.is_file() for path in test_artifacts), "Test artifacts should exist but will not be loaded."

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hashes_before = {name: sha256_file(path) for name, path in protected_artifacts.items()}
print(f"Repository: {ROOT}")
print(f"Protected artifacts fingerprinted: {len(hashes_before)}")
print("Test artifacts exist; their contents were not loaded.")

Repository: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1
Protected artifacts fingerprinted: 11
Test artifacts exist; their contents were not loaded.


### What this cell does
Loads and validates the existing tree-preprocessed train and validation matrices, their targets, validation identifiers, and split-level reference counts.

### Why it matters
All candidates must see identical finite features and correctly aligned labels; preprocessing must not be refitted during tuning.

### What to understand
Passing checks confirm 57,776 training rows and 6,640 validation rows with the established 360-feature schema and class distributions.

In [2]:
X_train = pd.read_csv(input_paths["X_train_tree"])
X_validation = pd.read_csv(input_paths["X_validation_tree"])
y_train_frame = pd.read_csv(input_paths["y_train"])
y_validation_frame = pd.read_csv(input_paths["y_validation"])
validation_identifiers = pd.read_csv(input_paths["validation_identifiers"])
baseline_metrics = pd.read_csv(input_paths["baseline_metrics"])
split_summary = pd.read_csv(input_paths["final_split_summary"]).set_index("split")
TARGET = "slowdown_in_5min"
assert y_train_frame.columns.tolist() == [TARGET] and y_validation_frame.columns.tolist() == [TARGET]
y_train = y_train_frame[TARGET].astype(int)
y_validation = y_validation_frame[TARGET].astype(int)

assert X_train.shape == (57_776, 360) and X_validation.shape == (6_640, 360)
assert X_train.columns.tolist() == X_validation.columns.tolist()
assert len(y_train) == len(X_train) and len(y_validation) == len(X_validation) == len(validation_identifiers)
assert set(y_train.unique()) == {0, 1} and set(y_validation.unique()) == {0, 1}
assert not X_train.isna().any().any() and not X_validation.isna().any().any()
assert np.isfinite(X_train.to_numpy(dtype=float)).all() and np.isfinite(X_validation.to_numpy(dtype=float)).all()
assert int(y_train.sum()) == int(split_summary.loc["train", "positive_rows"])
assert int(y_validation.sum()) == int(split_summary.loc["validation", "positive_rows"])
assert len(y_train) == int(split_summary.loc["train", "total_rows"])
assert len(y_validation) == int(split_summary.loc["validation", "total_rows"])
print(f"Train: {X_train.shape}, positive={y_train.sum():,} ({y_train.mean():.2%})")
print(f"Validation: {X_validation.shape}, positive={y_validation.sum():,} ({y_validation.mean():.2%})")
print(f"Recorded test positive rate (summary only, no test rows loaded): {split_summary.loc['test', 'positive_rate']:.2f}%")

Train: (57776, 360), positive=16,605 (28.74%)
Validation: (6640, 360), positive=4,328 (65.18%)
Recorded test positive rate (summary only, no test rows loaded): 24.03%


### What this cell does
Extracts the existing threshold-0.50 LightGBM and Random Forest validation metrics as immutable baseline references.

### Why it matters
Tuned results must be compared against the actual prior experiment rather than a reconstructed or overwritten baseline.

### What to understand
The displayed rows define the PR-AUC, ROC-AUC, precision, recall, F1, false-positive, and false-negative values tuning must improve or trade off.

In [3]:
baseline_reference = baseline_metrics.loc[
    baseline_metrics["model"].isin(["LightGBM", "RandomForest"])
].copy().set_index("model")
assert set(baseline_reference.index) == {"LightGBM", "RandomForest"}
assert baseline_reference["threshold"].eq(0.50).all()
display(baseline_reference[["pr_auc", "roc_auc", "precision", "recall", "f1", "tn", "fp", "fn", "tp"]])

,pr_auc,roc_auc,precision,recall,f1,tn,fp,fn,tp
model,,,,,,,,,
LightGBM,0.986832,0.970864,0.960271,0.915896,0.937559,2148,164,364,3964
RandomForest,0.983162,0.962840,0.982732,0.867837,0.921718,2246,66,572,3756


### What this cell does
Defines reproducible candidate generation and metric helpers shared by both searches, including a transparent overfitting flag.

### Why it matters
A single evaluation implementation prevents metric drift, while fixed seeds and serialized parameter values make every trial reproducible.

### What to understand
Candidates are ranked primarily by validation average precision; F1 only breaks an exact PR-AUC tie, and threshold optimization is not performed.

In [4]:
RANDOM_SEED = 42
THRESHOLD = 0.50
SEARCH_N_JOBS = min(4, max(1, (os.cpu_count() or 2) // 2))
SEARCH_METHOD = "seeded custom randomized search on fixed validation PR-AUC"

def python_value(value):
    return value.item() if isinstance(value, np.generic) else value

def clean_parameters(parameters):
    return {key: python_value(value) for key, value in parameters.items()}

def candidate_list(baseline_parameters, search_space, total_candidates, seed):
    candidates = [clean_parameters(baseline_parameters)]
    seen = {json.dumps(candidates[0], sort_keys=True)}
    sampled = ParameterSampler(search_space, n_iter=min(total_candidates * 4, np.prod([len(v) for v in search_space.values()])), random_state=seed)
    for parameters in sampled:
        cleaned = clean_parameters(parameters)
        key = json.dumps(cleaned, sort_keys=True)
        if key not in seen:
            candidates.append(cleaned); seen.add(key)
        if len(candidates) == total_candidates:
            break
    assert len(candidates) == total_candidates
    return candidates

def binary_metrics(y_true, probability, threshold=THRESHOLD):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "pr_auc": average_precision_score(y_true, probability),
        "roc_auc": roc_auc_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

print(f"Search method: {SEARCH_METHOD}")
print(f"Worker threads per fitted model: {SEARCH_N_JOBS}; searches themselves are sequential.")

Search method: seeded custom randomized search on fixed validation PR-AUC
Worker threads per fitted model: 4; searches themselves are sequential.


### What this cell does
Tests 20 bounded LightGBM configurations covering capacity, learning rate, sampling, regularization, and optional training-derived class weighting.

### Why it matters
LightGBM is tuned first because it led the baseline comparison, while the compact search avoids an impractical exhaustive grid.

### What to understand
Each candidate is fitted only on train and ranked only by validation PR-AUC. A scale weight of 1 means no weighting; alternative values come solely from the training label ratio.

In [5]:
LIGHTGBM_CANDIDATE_COUNT = 20
train_imbalance_ratio = float((y_train == 0).sum() / (y_train == 1).sum())
lightgbm_baseline_parameters = {
    "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31, "max_depth": -1,
    "min_child_samples": 20, "subsample": 1.0, "colsample_bytree": 1.0,
    "reg_alpha": 0.0, "reg_lambda": 0.0, "scale_pos_weight": train_imbalance_ratio,
}
lightgbm_space = {
    "n_estimators": [200, 300, 500, 700], "learning_rate": [0.02, 0.03, 0.05, 0.08, 0.10],
    "num_leaves": [15, 31, 63, 127], "max_depth": [-1, 6, 10, 14],
    "min_child_samples": [10, 20, 40, 80], "subsample": [0.70, 0.85, 1.0],
    "colsample_bytree": [0.60, 0.80, 1.0], "reg_alpha": [0.0, 0.1, 1.0, 5.0],
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
    "scale_pos_weight": [1.0, float(np.sqrt(train_imbalance_ratio)), train_imbalance_ratio],
}
lightgbm_candidates = candidate_list(lightgbm_baseline_parameters, lightgbm_space, LIGHTGBM_CANDIDATE_COUNT, RANDOM_SEED)
lightgbm_trial_rows, best_lightgbm, best_lightgbm_key = [], None, (-np.inf, -np.inf)
lightgbm_search_started = time.perf_counter()
for trial_number, parameters in enumerate(lightgbm_candidates, start=1):
    started = time.perf_counter()
    model = LGBMClassifier(objective="binary", random_state=RANDOM_SEED, n_jobs=SEARCH_N_JOBS,
                           verbosity=-1, subsample_freq=1, **parameters)
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_validation)[:, 1]
    metrics = binary_metrics(y_validation, probability)
    duration = time.perf_counter() - started
    lightgbm_trial_rows.append({"model": "LightGBM", "trial_number": trial_number,
                                "is_baseline_configuration": trial_number == 1,
                                "parameters_json": json.dumps(parameters, sort_keys=True),
                                **metrics, "execution_time_seconds": duration, "status": "success"})
    ranking_key = (metrics["pr_auc"], metrics["f1"])
    if ranking_key > best_lightgbm_key:
        best_lightgbm_key, best_lightgbm = ranking_key, model
        best_lightgbm_parameters, best_lightgbm_validation_probability = parameters, probability.copy()
    print(f"LightGBM {trial_number:02d}/{LIGHTGBM_CANDIDATE_COUNT}: PR-AUC={metrics['pr_auc']:.6f}, F1={metrics['f1']:.4f}, {duration:.2f}s")
lightgbm_search_seconds = time.perf_counter() - lightgbm_search_started
print(f"Best LightGBM validation PR-AUC: {best_lightgbm_key[0]:.6f}")
print(json.dumps(best_lightgbm_parameters, indent=2, sort_keys=True))

LightGBM 01/20: PR-AUC=0.986832, F1=0.9376, 3.52s


LightGBM 02/20: PR-AUC=0.981368, F1=0.9086, 4.75s


LightGBM 03/20: PR-AUC=0.983742, F1=0.9347, 2.16s


LightGBM 04/20: PR-AUC=0.979504, F1=0.9099, 4.36s


LightGBM 05/20: PR-AUC=0.982894, F1=0.9124, 6.14s


...

LightGBM 16/20: PR-AUC=0.981953, F1=0.9225, 3.48s


LightGBM 17/20: PR-AUC=0.981490, F1=0.9125, 7.37s


LightGBM 18/20: PR-AUC=0.981829, F1=0.9132, 12.76s


LightGBM 19/20: PR-AUC=0.984596, F1=0.9246, 2.53s


LightGBM 20/20: PR-AUC=0.983734, F1=0.9182, 7.82s
Best LightGBM validation PR-AUC: 0.986832
{
  "colsample_bytree": 1.0,
  "learning_rate": 0.05,
  "max_depth": -1,
  "min_child_samples": 20,
  "n_estimators": 300,
  "num_leaves": 31,
  "reg_alpha": 0.0,
  "reg_lambda": 0.0,
  "scale_pos_weight": 2.4794339054501657,
  "subsample": 1.0
}


### What this cell does
Tests 12 bounded Random Forest configurations spanning tree count, depth, split constraints, feature sampling, bootstrap behavior, and optional class weighting.

### Why it matters
Random Forest is tuned independently so its validation ranking is not influenced by LightGBM choices, and limited parallelism avoids monopolizing the machine.

### What to understand
The first candidate reproduces the baseline configuration; all candidates use the same train and validation rows and the same PR-AUC objective.

In [6]:
RANDOM_FOREST_CANDIDATE_COUNT = 12
random_forest_baseline_parameters = {
    "n_estimators": 300, "max_depth": None, "min_samples_split": 2,
    "min_samples_leaf": 1, "max_features": "sqrt", "bootstrap": True,
    "class_weight": "balanced",
}
random_forest_space = {
    "n_estimators": [200, 300, 500], "max_depth": [None, 12, 20, 30, 40],
    "min_samples_split": [2, 5, 10, 20], "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.5], "bootstrap": [True, False],
    "class_weight": [None, "balanced", "balanced_subsample"],
}
random_forest_candidates = candidate_list(random_forest_baseline_parameters, random_forest_space, RANDOM_FOREST_CANDIDATE_COUNT, RANDOM_SEED + 1)
random_forest_trial_rows, best_random_forest, best_random_forest_key = [], None, (-np.inf, -np.inf)
random_forest_search_started = time.perf_counter()
for trial_number, parameters in enumerate(random_forest_candidates, start=1):
    started = time.perf_counter()
    model = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=SEARCH_N_JOBS, **parameters)
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_validation)[:, 1]
    metrics = binary_metrics(y_validation, probability)
    duration = time.perf_counter() - started
    random_forest_trial_rows.append({"model": "RandomForest", "trial_number": trial_number,
                                    "is_baseline_configuration": trial_number == 1,
                                    "parameters_json": json.dumps(parameters, sort_keys=True),
                                    **metrics, "execution_time_seconds": duration, "status": "success"})
    ranking_key = (metrics["pr_auc"], metrics["f1"])
    if ranking_key > best_random_forest_key:
        best_random_forest_key, best_random_forest = ranking_key, model
        best_random_forest_parameters, best_random_forest_validation_probability = parameters, probability.copy()
    print(f"RandomForest {trial_number:02d}/{RANDOM_FOREST_CANDIDATE_COUNT}: PR-AUC={metrics['pr_auc']:.6f}, F1={metrics['f1']:.4f}, {duration:.2f}s")
random_forest_search_seconds = time.perf_counter() - random_forest_search_started
print(f"Best Random Forest validation PR-AUC: {best_random_forest_key[0]:.6f}")
print(json.dumps(best_random_forest_parameters, indent=2, sort_keys=True))

RandomForest 01/12: PR-AUC=0.983162, F1=0.9217, 16.20s


RandomForest 02/12: PR-AUC=0.979268, F1=0.9187, 30.94s


RandomForest 03/12: PR-AUC=0.980419, F1=0.9215, 223.13s


RandomForest 04/12: PR-AUC=0.982302, F1=0.9283, 10.31s


RandomForest 05/12: PR-AUC=0.980395, F1=0.9159, 194.25s


...

RandomForest 08/12: PR-AUC=0.982648, F1=0.9066, 11.28s


RandomForest 09/12: PR-AUC=0.981154, F1=0.9305, 505.30s


RandomForest 10/12: PR-AUC=0.981563, F1=0.9212, 18.59s


RandomForest 11/12: PR-AUC=0.979723, F1=0.9143, 19.02s


RandomForest 12/12: PR-AUC=0.975293, F1=0.9212, 250.23s
Best Random Forest validation PR-AUC: 0.983162
{
  "bootstrap": true,
  "class_weight": "balanced",
  "max_depth": null,
  "max_features": "sqrt",
  "min_samples_leaf": 1,
  "min_samples_split": 2,
  "n_estimators": 300
}


### What this cell does
Evaluates each selected configuration on both train and validation, compares it with its saved baseline, and applies an explicit train–validation PR-AUC gap warning.

### Why it matters
Near-perfect training performance can hide overfitting; validation performance remains the selection criterion even when train scores are higher.

### What to understand
An overfitting warning is raised when the PR-AUC gap is at least 0.03, or when train PR-AUC is at least 0.995 with a gap of at least 0.01.

In [7]:
selected_models = {
    "LightGBM": {"model": best_lightgbm, "parameters": best_lightgbm_parameters,
                  "validation_probability": best_lightgbm_validation_probability,
                  "candidate_count": LIGHTGBM_CANDIDATE_COUNT, "search_seconds": lightgbm_search_seconds},
    "RandomForest": {"model": best_random_forest, "parameters": best_random_forest_parameters,
                       "validation_probability": best_random_forest_validation_probability,
                       "candidate_count": RANDOM_FOREST_CANDIDATE_COUNT, "search_seconds": random_forest_search_seconds},
}
summary_rows, long_comparison_rows = [], []
for model_name, information in selected_models.items():
    train_probability = information["model"].predict_proba(X_train)[:, 1]
    train_metrics = binary_metrics(y_train, train_probability)
    validation_metrics = binary_metrics(y_validation, information["validation_probability"])
    baseline = baseline_reference.loc[model_name]
    pr_auc_gap = train_metrics["pr_auc"] - validation_metrics["pr_auc"]
    overfitting_warning = bool(pr_auc_gap >= 0.03 or (train_metrics["pr_auc"] >= 0.995 and pr_auc_gap >= 0.01))
    information["train_probability"] = train_probability
    information["train_metrics"] = train_metrics
    information["validation_metrics"] = validation_metrics
    information["overfitting_warning"] = overfitting_warning
    summary_rows.append({
        "model": model_name, "search_method": SEARCH_METHOD,
        "candidates_tested": information["candidate_count"],
        "search_time_seconds": information["search_seconds"],
        "best_parameters_json": json.dumps(information["parameters"], sort_keys=True),
        "baseline_validation_pr_auc": baseline["pr_auc"], "tuned_validation_pr_auc": validation_metrics["pr_auc"],
        "pr_auc_improvement": validation_metrics["pr_auc"] - baseline["pr_auc"],
        "baseline_validation_roc_auc": baseline["roc_auc"], "tuned_validation_roc_auc": validation_metrics["roc_auc"],
        "tuned_train_pr_auc": train_metrics["pr_auc"], "train_validation_pr_auc_gap": pr_auc_gap,
        "tuned_validation_precision": validation_metrics["precision"],
        "tuned_validation_recall": validation_metrics["recall"], "tuned_validation_f1": validation_metrics["f1"],
        "tuned_validation_tn": validation_metrics["tn"], "tuned_validation_fp": validation_metrics["fp"],
        "tuned_validation_fn": validation_metrics["fn"], "tuned_validation_tp": validation_metrics["tp"],
        "tuning_improved_pr_auc": validation_metrics["pr_auc"] > baseline["pr_auc"] + 1e-12,
        "overfitting_warning": overfitting_warning,
        "train_positive_rate": y_train.mean(), "validation_positive_rate": y_validation.mean(),
    })
    for variant, metrics in [("baseline_validation", baseline), ("tuned_train", train_metrics), ("tuned_validation", validation_metrics)]:
        long_comparison_rows.append({"model": model_name, "variant": variant, **{key: metrics[key] for key in ["pr_auc", "roc_auc", "precision", "recall", "f1", "tn", "fp", "fn", "tp"]}})
tuning_summary = pd.DataFrame(summary_rows).sort_values("tuned_validation_pr_auc", ascending=False).reset_index(drop=True)
baseline_tuned_comparison = pd.DataFrame(long_comparison_rows)
display(tuning_summary)
display(baseline_tuned_comparison)

,model,search_method,candidates_tested,search_time_seconds,best_parameters_json,baseline_validation_pr_auc,tuned_validation_pr_auc,pr_auc_improvement,baseline_validation_roc_auc,tuned_validation_roc_auc,...,tuned_validation_recall,tuned_validation_f1,tuned_validation_tn,tuned_validation_fp,tuned_validation_fn,tuned_validation_tp,tuning_improved_pr_auc,overfitting_warning,train_positive_rate,validation_positive_rate
0,LightGBM,seeded custom randomized search on fixed valid...,20,112.014288,"{""colsample_bytree"": 1.0, ""learning_rate"": 0.0...",0.986832,0.986832,0.000000e+00,0.970864,0.970864,...,0.915896,0.937559,2148,164,364,3964,False,True,0.287403,0.651807
1,RandomForest,seeded custom randomized search on fixed valid...,12,1885.740682,"{""bootstrap"": true, ""class_weight"": ""balanced""...",0.983162,0.983162,-1.110223e-16,0.962840,0.962840,...,0.867837,0.921718,2246,66,572,3756,False,True,0.287403,0.651807


,model,variant,pr_auc,roc_auc,precision,recall,f1,tn,fp,fn,tp
0,LightGBM,baseline_validation,0.986832,0.970864,0.960271,0.915896,0.937559,2148,164,364,3964
1,LightGBM,tuned_train,0.999999,1.000000,0.998017,1.000000,0.999007,41138,33,0,16605
2,LightGBM,tuned_validation,0.986832,0.970864,0.960271,0.915896,0.937559,2148,164,364,3964
3,RandomForest,baseline_validation,0.983162,0.962840,0.982732,0.867837,0.921718,2246,66,572,3756
4,RandomForest,tuned_train,1.000000,1.000000,0.999940,1.000000,0.999970,41170,1,0,16605
5,RandomForest,tuned_validation,0.983162,0.962840,0.982732,0.867837,0.921718,2246,66,572,3756


### What this cell does
Creates row-aligned validation probabilities and threshold-0.50 classes for both tuned models while retaining all available identifiers separately from features.

### Why it matters
This report supports later error and threshold analysis without exposing the test set or mixing identifiers into model fitting.

### What to understand
Every probability belongs to the validation identifier on the same row; no optimized threshold or calibrated probability is included.

In [8]:
tuned_validation_predictions = validation_identifiers.reset_index(drop=True).copy()
tuned_validation_predictions["true_target"] = y_validation.to_numpy()
tuned_validation_predictions["lightgbm_tuned_probability"] = best_lightgbm_validation_probability
tuned_validation_predictions["lightgbm_tuned_class_0_50"] = (best_lightgbm_validation_probability >= THRESHOLD).astype(int)
tuned_validation_predictions["random_forest_tuned_probability"] = best_random_forest_validation_probability
tuned_validation_predictions["random_forest_tuned_class_0_50"] = (best_random_forest_validation_probability >= THRESHOLD).astype(int)
assert len(tuned_validation_predictions) == len(y_validation)
display(tuned_validation_predictions.head())

,machine_id,run_id,segment_id,timestamp,valid_5min_horizon,valid_10min_horizon,slowdown_in_10min,true_target,lightgbm_tuned_probability,lightgbm_tuned_class_0_50,random_forest_tuned_probability,random_forest_tuned_class_0_50
0,7232bc533c21ce408d45d473,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,7232bc533c21ce408d45d473__1e11f9c7-7d25-4a5d-a...,2026-08-03T20:39:06.889000Z,True,True,1.0,1,0.794404,1,0.583333,1
1,7232bc533c21ce408d45d473,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,7232bc533c21ce408d45d473__1e11f9c7-7d25-4a5d-a...,2026-08-03T20:39:08.223000Z,True,True,1.0,1,0.889264,1,0.533333,1
2,7232bc533c21ce408d45d473,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,7232bc533c21ce408d45d473__1e11f9c7-7d25-4a5d-a...,2026-08-03T20:39:10.906000Z,True,True,1.0,1,0.891733,1,0.440000,0
3,7232bc533c21ce408d45d473,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,7232bc533c21ce408d45d473__1e11f9c7-7d25-4a5d-a...,2026-08-03T20:39:12.370000Z,True,True,1.0,1,0.761646,1,0.453333,0
4,7232bc533c21ce408d45d473,1e11f9c7-7d25-4a5d-ae2a-f9bc6e91415d,7232bc533c21ce408d45d473__1e11f9c7-7d25-4a5d-a...,2026-08-03T20:39:15.278000Z,True,True,1.0,1,0.438236,0,0.446667,0


### What this cell does
Saves the two tuned models, their exact parameter dictionaries, every candidate result, the tuning summary, and validation predictions.

### Why it matters
Separate tuned paths protect baseline estimators, while complete trial records make the bounded search auditable and reproducible.

### What to understand
The saved models remain train-only fits selected using validation PR-AUC; they are not final or production models.

In [9]:
tuning_results = pd.DataFrame(lightgbm_trial_rows + random_forest_trial_rows)
tuning_results["selected_best"] = False
for model_name, information in selected_models.items():
    matching = tuning_results["model"].eq(model_name) & tuning_results["parameters_json"].eq(json.dumps(information["parameters"], sort_keys=True))
    tuning_results.loc[matching, "selected_best"] = True
tuning_results_path = REPORT_DIR / "model_tuning_results.csv"
tuning_summary_path = REPORT_DIR / "model_tuning_summary.csv"
predictions_path = REPORT_DIR / "tuned_validation_predictions.csv"
tuning_results.to_csv(tuning_results_path, index=False)
tuning_summary.to_csv(tuning_summary_path, index=False)
tuned_validation_predictions.to_csv(predictions_path, index=False)

model_paths = {
    "LightGBM": TUNED_MODEL_DIR / "tuned_lightgbm.joblib",
    "RandomForest": TUNED_MODEL_DIR / "tuned_random_forest.joblib",
}
parameter_paths = {
    "LightGBM": TUNED_MODEL_DIR / "lightgbm_best_params.json",
    "RandomForest": TUNED_MODEL_DIR / "random_forest_best_params.json",
}
for model_name, information in selected_models.items():
    joblib.dump(information["model"], model_paths[model_name])
    parameter_paths[model_name].write_text(json.dumps(information["parameters"], indent=2, sort_keys=True) + "\n")
print(f"Saved {len(tuning_results)} candidate rows to {tuning_results_path}")
print(f"Saved summary: {tuning_summary_path}")
print(f"Saved validation predictions: {predictions_path}")
for name in model_paths: print(f"Saved {name}: {model_paths[name]} and {parameter_paths[name]}")

Saved 32 candidate rows to /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/model_tuning_results.csv
Saved summary: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/model_tuning_summary.csv
Saved validation predictions: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/tuned_validation_predictions.csv
Saved LightGBM: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/tuned/tuned_lightgbm.joblib and /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/tuned/lightgbm_best_params.json
Saved RandomForest: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/tuned/tuned_random_forest.joblib and /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/tuned/random_forest_best_params.json


### What this cell does
Saves tuned-model precision–recall and ROC curves, a baseline-versus-tuned metric chart, and a candidate-history plot.

### Why it matters
Visual comparisons expose ranking quality and search stability without selecting a threshold or evaluating the test set.

### What to understand
The curves use validation probabilities only; the metric bars compare saved baseline values with tuned validation values.

In [10]:
figure_paths = []
fig, ax = plt.subplots(figsize=(8, 6))
for model_name, information in selected_models.items():
    precision_values, recall_values, _ = precision_recall_curve(y_validation, information["validation_probability"])
    score = information["validation_metrics"]["pr_auc"]
    ax.plot(recall_values, precision_values, linewidth=2, label=f"{model_name} tuned (PR-AUC={score:.4f})")
ax.axhline(y_validation.mean(), color="gray", linestyle="--", label=f"Validation prevalence={y_validation.mean():.3f}")
ax.set(xlabel="Recall", ylabel="Precision", title="Tuned-model validation precision–recall curves", xlim=(0, 1), ylim=(0, 1.02))
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
path = FIGURE_DIR / "tuned_precision_recall_curves.png"; fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig); figure_paths.append(path)

fig, ax = plt.subplots(figsize=(8, 6))
for model_name, information in selected_models.items():
    fpr, tpr, _ = roc_curve(y_validation, information["validation_probability"])
    score = information["validation_metrics"]["roc_auc"]
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} tuned (ROC-AUC={score:.4f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="Tuned-model validation ROC curves", xlim=(0, 1), ylim=(0, 1.02))
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
path = FIGURE_DIR / "tuned_roc_curves.png"; fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig); figure_paths.append(path)

metric_names = ["pr_auc", "roc_auc", "f1", "recall", "precision"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, model_name in zip(axes, ["LightGBM", "RandomForest"]):
    baseline_values = [baseline_reference.loc[model_name, metric] for metric in metric_names]
    tuned_values = [selected_models[model_name]["validation_metrics"][metric] for metric in metric_names]
    positions = np.arange(len(metric_names)); width = 0.36
    ax.bar(positions - width / 2, baseline_values, width, label="Baseline")
    ax.bar(positions + width / 2, tuned_values, width, label="Tuned")
    ax.set_xticks(positions, metric_names, rotation=25); ax.set_ylim(0, 1.03); ax.set_title(model_name); ax.grid(axis="y", alpha=0.25); ax.legend()
fig.suptitle("Baseline versus tuned validation metrics at threshold 0.50"); fig.tight_layout()
path = FIGURE_DIR / "baseline_vs_tuned_metrics.png"; fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig); figure_paths.append(path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name in zip(axes, ["LightGBM", "RandomForest"]):
    rows = tuning_results.loc[tuning_results["model"].eq(model_name)].sort_values("trial_number")
    ax.plot(rows["trial_number"], rows["pr_auc"], marker="o", linewidth=1.5)
    ax.axhline(baseline_reference.loc[model_name, "pr_auc"], color="gray", linestyle="--", label="Saved baseline")
    ax.set(xlabel="Candidate number", ylabel="Validation PR-AUC", title=f"{model_name} search history"); ax.grid(alpha=0.25); ax.legend()
fig.tight_layout(); path = FIGURE_DIR / "tuning_search_history.png"; fig.savefig(path, dpi=160, bbox_inches="tight"); plt.close(fig); figure_paths.append(path)
print(f"Saved {len(figure_paths)} figures under {FIGURE_DIR}")

Saved 4 figures under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/model_tuning


### What this cell does
Reloads all tuning outputs, verifies protected checksums, and prints the final bounded-search conclusions and explicit stop condition.

### Why it matters
The handoff must prove reproducibility and clearly separate validation-based tuning candidates from any future final model decision.

### What to understand
High validation scores remain provisional because validation has 65.18% positives versus 28.74% in train and 24.03% in test.

In [11]:
saved_results = pd.read_csv(tuning_results_path)
saved_summary = pd.read_csv(tuning_summary_path)
saved_predictions = pd.read_csv(predictions_path)
assert len(saved_results) == LIGHTGBM_CANDIDATE_COUNT + RANDOM_FOREST_CANDIDATE_COUNT
assert saved_results.groupby("model")["selected_best"].sum().eq(1).all()
assert len(saved_summary) == 2 and len(saved_predictions) == len(y_validation)
assert all(path.exists() and path.stat().st_size > 0 for path in [*model_paths.values(), *parameter_paths.values(), *figure_paths])
for model_name, path in model_paths.items():
    reloaded = joblib.load(path)
    assert hasattr(reloaded, "predict_proba")
hashes_after = {name: sha256_file(path) for name, path in protected_artifacts.items()}
protected_unchanged = hashes_before == hashes_after
assert protected_unchanged, "An input, preprocessing artifact, or baseline model was modified."

models_for_threshold_review = tuning_summary.loc[tuning_summary["tuning_improved_pr_auc"], "model"].tolist()
if not models_for_threshold_review:
    models_for_threshold_review = tuning_summary.head(1)["model"].tolist()
print("FINAL MODEL-TUNING REPORT")
print(f"Search method: {SEARCH_METHOD}")
print(f"Candidates tested: LightGBM={LIGHTGBM_CANDIDATE_COUNT}, RandomForest={RANDOM_FOREST_CANDIDATE_COUNT}")
for row in tuning_summary.itertuples(index=False):
    print(f"{row.model}: baseline PR-AUC={row.baseline_validation_pr_auc:.6f}, tuned PR-AUC={row.tuned_validation_pr_auc:.6f}, delta={row.pr_auc_improvement:+.6f}")
    print(f"  validation precision={row.tuned_validation_precision:.4f}, recall={row.tuned_validation_recall:.4f}, F1={row.tuned_validation_f1:.4f}, FP={row.tuned_validation_fp}, FN={row.tuned_validation_fn}")
    print(f"  train PR-AUC={row.tuned_train_pr_auc:.6f}, gap={row.train_validation_pr_auc_gap:.6f}, overfitting warning={row.overfitting_warning}")
    print(f"  best parameters={row.best_parameters_json}")
print(f"Models recommended to proceed to a separate threshold-optimization review: {models_for_threshold_review}")
print("WARNING: validation positive rate is 65.18%, versus 28.74% in train and 24.03% in test.")
print("High validation performance must not be treated as final production performance.")
print(f"All protected inputs, preprocessing objects, and baseline models unchanged: {protected_unchanged}")
print("STOP: no threshold optimization, calibration, train+validation refit, test evaluation, SHAP, or final model selection was performed.")

FINAL MODEL-TUNING REPORT
Search method: seeded custom randomized search on fixed validation PR-AUC
Candidates tested: LightGBM=20, RandomForest=12
LightGBM: baseline PR-AUC=0.986832, tuned PR-AUC=0.986832, delta=+0.000000
  validation precision=0.9603, recall=0.9159, F1=0.9376, FP=164, FN=364
  train PR-AUC=0.999999, gap=0.013168, overfitting warning=True
  best parameters={"colsample_bytree": 1.0, "learning_rate": 0.05, "max_depth": -1, "min_child_samples": 20, "n_estimators": 300, "num_leaves": 31, "reg_alpha": 0.0, "reg_lambda": 0.0, "scale_pos_weight": 2.4794339054501657, "subsample": 1.0}
RandomForest: baseline PR-AUC=0.983162, tuned PR-AUC=0.983162, delta=-0.000000
  validation precision=0.9827, recall=0.8678, F1=0.9217, FP=66, FN=572
  train PR-AUC=1.000000, gap=0.016838, overfitting warning=True
  best parameters={"bootstrap": true, "class_weight": "balanced", "max_depth": null, "max_features": "sqrt", "min_samples_leaf": 1, "min_samples_split": 2, "n_estimators": 300}
Models 